In [1]:
import json
import pathlib
import re

import numpy as np
import pandas as pd
import swifter

from tqdm.auto import tqdm

from iupac_name_parser import IUPACNameParser

from rdkit import Chem, RDLogger

RDLogger.DisableLog('rdApp.*')

In [2]:
REPLACE_STORED_RESULTS = False

In [3]:
ROOT_DIR = pathlib.Path.cwd()
RAW_DATA_DIR = ROOT_DIR / "raw_data"
TREATED_DATA_DIR = ROOT_DIR / "treated_data"

RESULTS_DIR = TREATED_DATA_DIR / "rotation_results"
assert RESULTS_DIR.exists()

EXTRACTED_DIR = RAW_DATA_DIR / "extracted"
assert EXTRACTED_DIR.exists()

DATA_BLOCKS_DIR = TREATED_DATA_DIR / "data_blocks"
assert DATA_BLOCKS_DIR.exists()

DATA_FOR_CURATION_DIR = TREATED_DATA_DIR / "for_llm_curation" / "rotation"
DATA_FOR_CURATION_DIR.mkdir(parents=True, exist_ok=True)

COLLECTED_BOLD_BLOCKS_PATH = TREATED_DATA_DIR / "collected_bold_blocks.csv.xz"
RESULTS_FROM_LLM_PATH = TREATED_DATA_DIR / "rotation_results_from_llm.csv.xz"
RESULTS_FROM_PATTERNS_PATH = TREATED_DATA_DIR / "rotation_results_from_patterns.csv.xz"
results_FOR_LLM_CURATION_PATH = (
    TREATED_DATA_DIR / "rotation_results_for_curation.csv.xz"
)

In [4]:
mol_parser = IUPACNameParser()

In [5]:
pd.set_option("display.max_columns", None)


def head(df, n=2):
    display(df.head(n))
    print(f"Contains {len(df)} rows")


def get_extended_block(
    block: dict, molecule: str, article_id: str, file_stem: str
) -> dict | None:
    try:
        name, smiles, appears_ambiguous, stereo_ignored = mol_parser.to_smiles(molecule)
    except Exception:
        name = "unknown"
        smiles = None
        appears_ambiguous = stereo_ignored = False
    return {
        "article_id": article_id,
        "file_stem": file_stem,
        **block,
        "molecule": name,
        "smiles": smiles,
        "appears_ambiguous": appears_ambiguous,
        "stereo_ignored": stereo_ignored,
    }


def canonical_smiles(smiles: str | None | float) -> str | None:
    if smiles is None or isinstance(smiles, float):
        return None
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return None
    return Chem.MolToSmiles(mol, isomericSmiles=True)

In [6]:
SHORT_THRESHOLD_DISTANCE = 200
LONG_THRESHOLD_DISTANCE = 800

UNITS = r"(?:cm\s*-1|ppm)"
UNITS_PATTERN = re.compile(rf"\(?{UNITS}\)?")

UNSIGNED_REAL = r"\d+(?:\.\d+)?"
NUMBER_OR_RANGE = rf"{UNSIGNED_REAL}(?:\s*-\s*{UNSIGNED_REAL})?"
INSIDE_PARENTHESES = r"\((?:[^()]+)\)"
NMR_PEAK = rf"{NUMBER_OR_RANGE}(?:\s*{INSIDE_PARENTHESES})?"
NMR_PEAK_LIST = rf"{NMR_PEAK}(?:[\s,]+{NMR_PEAK})*"
NMR_PATTERN = re.compile(
    rf"""
        (?:\d*[HCF]\s*)?               # Optional number and H/C/F notation
        NMR                            # NMR literal
        (?:\s*{INSIDE_PARENTHESES})+   # Parenthesized data
        \s*                            # Optional whitespace
        [δ=:\s]*                       # Optional NMR notation
        {NMR_PEAK_LIST}                # NMR peak list
    """,
    re.VERBOSE,
)

HRMS_PATTERN = re.compile(
    rf"HRMS(.{{1,100}})?found[\s:]+{UNSIGNED_REAL}", re.DOTALL | re.IGNORECASE
)

COMMA_SEPARATED_NUMBERS = rf"{UNSIGNED_REAL}(?:\s*,\s*{UNSIGNED_REAL})+"
IR_PATTERN = re.compile(
    rf"(?:FT)?IR[^\d]{{1,20}}{COMMA_SEPARATED_NUMBERS}",
    re.IGNORECASE,
)

SEPARATORS_AND_SPACES_PATTERN = re.compile(r"[,;\.][,;\.\s]+")


def remove_analysis_data(text: str) -> str:
    text = UNITS_PATTERN.sub("", text)
    text = NMR_PATTERN.sub("", text)
    text = HRMS_PATTERN.sub("", text)
    text = IR_PATTERN.sub("", text)
    text = SEPARATORS_AND_SPACES_PATTERN.sub("", text)
    return text.strip()


def retrieve_excess_or_ratio(
    rotation_block: dict,
    before_rotation_block: bool = False,
    data_blocks: list[dict] = None,
) -> str:
    article_id = rotation_block["article_id"]
    file_stem = rotation_block["file_stem"]
    if data_blocks is None:
        data_block_path = DATA_BLOCKS_DIR / article_id / f"{file_stem}.json"
        with open(data_block_path, "r") as f:
            data_blocks = json.load(f)
    if before_rotation_block:
        data_blocks = reversed(data_blocks)
    for data_block in data_blocks:
        if before_rotation_block:
            delta = rotation_block["stop"] - data_block["stop"]  # rot. block included
        else:
            delta = data_block["start"] - rotation_block["stop"]
        if delta > 0 and data_block["type"] == "rotation":
            return "unknown"
        if delta > 0 and data_block["type"] in ["excess", "ratio"]:
            if delta < SHORT_THRESHOLD_DISTANCE:
                return data_block["text"]
            if delta < LONG_THRESHOLD_DISTANCE:
                text_file = EXTRACTED_DIR / article_id / f"{file_stem}.txt"
                with open(text_file, "r") as f:
                    text = f.read()
                if before_rotation_block:
                    in_between = text[data_block["start"] : rotation_block["start"]]
                else:
                    in_between = text[rotation_block["stop"] : data_block["start"]]
                in_between = remove_analysis_data(in_between).strip()
                if len(in_between) < SHORT_THRESHOLD_DISTANCE:
                    return data_block["text"]
                return "unknown"
            return "unknown"
    return "unknown"

In [7]:
EXCESS_VALUE = re.compile(rf"(>=?\s*)?({UNSIGNED_REAL})\s*%")
RATIO_VALUES = re.compile(rf"(>=?\s*)?({UNSIGNED_REAL})\s*[:\/]\s*({UNSIGNED_REAL})")


def parse_excess(s: str) -> float | None:
    match = EXCESS_VALUE.search(s)
    if match is None:
        match = RATIO_VALUES.search(s)
        if match is None:
            return None
        c1 = float(match.group(2))
        c2 = float(match.group(3))
        if (c1 + c2) == 0:
            return None
        return round((abs(c1 - c2) / (c1 + c2)) * 100, 1)
    return round(float(match.group(2)), 1)

In [8]:
def retrieve_molecule(
    rotation_block: dict, bold_blocks: list[dict], full_text: str
) -> str:
    for bold_block in reversed(bold_blocks):
        delta = rotation_block["start"] - bold_block["stop"]
        if delta > 0:
            if delta < SHORT_THRESHOLD_DISTANCE:
                return bold_block["text"]
            if delta < LONG_THRESHOLD_DISTANCE:
                in_between = full_text[bold_block["stop"] : rotation_block["start"]]
                in_between = remove_analysis_data(in_between).strip()
                if len(in_between) < SHORT_THRESHOLD_DISTANCE:
                    return bold_block["text"]
            return "unknown"
    return "unknown"

In [9]:
def read_bold_blocks(article_id: str, file_stem: str) -> list[dict] | None:
    bold_block_path = EXTRACTED_DIR / article_id / f"{file_stem}.json"
    if not bold_block_path.exists() or bold_block_path.stat().st_size == 0:
        return None
    with open(bold_block_path, "r") as f:
        return json.load(f)


def read_full_text(article_id: str, file_stem: str) -> str:
    text_file = EXTRACTED_DIR / article_id / f"{file_stem}.txt"
    with open(text_file, "r") as f:
        return f.read()

In [10]:
def get_unique_label(row):
    return "_".join(map(str, [row["article_id"], row["file_stem"], row["start"]]))

## Collect boldface text blocks

In [11]:
if not REPLACE_STORED_RESULTS and COLLECTED_BOLD_BLOCKS_PATH.exists():
    bold_blocks_df = pd.read_csv(COLLECTED_BOLD_BLOCKS_PATH)
else:
    files = list(EXTRACTED_DIR.glob("**/*.json"))
    articles_with_bold_blocks = []
    for file in tqdm(files, desc="Processing bold block files"):
        if file.stat().st_size == 0:
            continue
        with open(file, "r") as f:
            blocks_from_file = json.load(f)
        for block in blocks_from_file:
            extended_block = get_extended_block(
                block, block["text"], file.parent.name, file.stem
            )
            if extended_block["smiles"] is None:
                continue
            articles_with_bold_blocks.append(extended_block)
    bold_blocks_df = pd.DataFrame(articles_with_bold_blocks)
    bold_blocks_df.to_csv(COLLECTED_BOLD_BLOCKS_PATH, index=False)

head(bold_blocks_df)

,article_id,file_stem,type,start,stop,text,molecule,smiles,appears_ambiguous,stereo_ignored
0,2057268,ja511728b_si_001,bold,4315,4370,"tetrahydro-2,5-methanobenzo[b]oxepine-4-carbox...","tetrahydro-2,5-methanobenzo[b]oxepine-4-carbox...",O1C2=C(C3C(CC1C3)C(=O)[O-])C=CC=C2,True,False
1,2057268,ja511728b_si_001,bold,5878,5999,"(-)-Methyl (2S,3R,4S,5S,10R)-5,10- dihydroxy-...","(-)-Methyl (2S,3R,4S,5S,10R)-5,10-dihydroxy-2,...",O[C@]12C3=C(O[C@]([C@H]([C@@H]1C(=O)OC)C1=CC=C...,False,True


Contains 384913 rows


### Extract rotation results from bold text and data blocks

In [12]:
if not REPLACE_STORED_RESULTS and RESULTS_FROM_PATTERNS_PATH.exists():
    results_from_patterns_df = pd.read_csv(RESULTS_FROM_PATTERNS_PATH)
else:
    results_from_patterns = []
    data_block_files = list(DATA_BLOCKS_DIR.glob("**/*.json"))
    for data_block_file in tqdm(data_block_files, desc="Processing data block files"):
        article_id = data_block_file.parent.name
        file_stem = data_block_file.stem

        bold_blocks = read_bold_blocks(article_id, file_stem)
        if bold_blocks is None:
            continue
        full_text = read_full_text(article_id, file_stem)

        with open(data_block_file, "r") as f:
            data_blocks = json.load(f)
        for data_block in data_blocks:
            if data_block["type"] != "rotation":
                continue
            molecule = retrieve_molecule(data_block, bold_blocks, full_text)
            if molecule == "unknown":
                continue
            data_block["excess_or_ratio"] = "unknown"
            extended_block = get_extended_block(
                data_block, molecule, article_id, file_stem
            )
            excess_or_ratio = retrieve_excess_or_ratio(
                extended_block, before_rotation_block=True, data_blocks=data_blocks
            )
            if excess_or_ratio == "unknown":
                excess_or_ratio = retrieve_excess_or_ratio(
                    extended_block, before_rotation_block=False, data_blocks=data_blocks
                )
            del extended_block["type"]
            extended_block["excess_or_ratio"] = excess_or_ratio
            extended_block["excess"] = parse_excess(excess_or_ratio)
            if not (
                extended_block["smiles"] is None and extended_block["excess"] is None
            ):
                results_from_patterns.append(extended_block)

    results_from_patterns_df = pd.DataFrame(results_from_patterns)
    results_from_patterns_df.to_csv(RESULTS_FROM_PATTERNS_PATH, index=False)

head(results_from_patterns_df)

,article_id,file_stem,start,stop,text,excess_or_ratio,molecule,smiles,appears_ambiguous,stereo_ignored,excess
0,2057268,ja511728b_si_001,13951,13985,"[α]D 26 = -75.37° (c = 0.1, CHCl3)",88% ee,"(-)-Methyl (1R,2R,3S,3aR,8bS)-3a-(4-bromopheny...",BrC1=CC=C(C=C1)[C@@]12OC3=C([C@@]1([C@@H]([C@@...,False,True,88.0
1,6139439,ol8b00721_si_001,28602,28633,"[α]D 25 = 47.5 (c = 1.0, CHCl3)",dr = 94:6,"(S)-3,3-difluoro-4-(((1S,2R)-2-hydroxy-1,2-dip...",CC1=CC=C(C=C1)S(=O)(=O)OC(=C)C([C@H](C1=CC=CC=...,False,False,88.0


Contains 58989 rows


## Collect rotation results from LLM

In [13]:
if not REPLACE_STORED_RESULTS and RESULTS_FROM_LLM_PATH.exists():
    results_from_llm_df = pd.read_csv(RESULTS_FROM_LLM_PATH)
else:
    results_from_llm = []
    results_files = list(RESULTS_DIR.glob("*"))
    for path in tqdm(results_files, desc="Processing articles"):
        if not (path.is_dir() and path.name.isdigit()):
            continue
        article_id = path.name
        for subdir in path.glob("*"):
            if not subdir.is_dir():
                continue
            stem = subdir.name
            for block_file in subdir.glob("*.json"):
                with open(block_file, "r") as f:
                    block = json.load(f)
                extended_block = get_extended_block(
                    block, block["molecule"], article_id, stem
                )
                del block["type"]
                extended_block["excess"] = parse_excess(block["excess_or_ratio"])
                if not (
                    extended_block["smiles"] is None
                    and extended_block["excess"] is None
                ):
                    results_from_llm.append(extended_block)

    results_from_llm_df = pd.DataFrame(results_from_llm)
    results_from_llm_df.to_csv(RESULTS_FROM_LLM_PATH, index=False)

head(results_from_llm_df)

,article_id,file_stem,type,start,stop,text,molecule,excess_or_ratio,smiles,appears_ambiguous,stereo_ignored,excess
0,2057268,ja511728b_si_001,rotation,13951,13985,"[α]D 26 = -75.37° (c = 0.1, CHCl3)","(-)-Methyl (1R,2R,3S,3aR,8bS)-3a-(4-bromopheny...",88% ee,BrC1=CC=C(C=C1)[C@@]12OC3=C([C@@]1([C@@H]([C@@...,False,True,88.0
1,2057268,ja511728b_si_001,rotation,9137,9172,"[α]D 26 = -83.017° (c = 0.2, CHCl3)","(-)-Methyl (2S,3R,4S,5S,10R)-2-(4-Bromophenyl)...",99% ee,BrC1=CC=C(C=C1)[C@]12[C@H]([C@@H]([C@](C3=C(O1...,False,True,99.0


Contains 109129 rows


## Generate data for further LLM curation

In [14]:
llm_df = results_from_llm_df.copy()
llm_df["label"] = llm_df.swifter.apply(get_unique_label, axis=1)
llm_df.set_index("label", inplace=True)
llm_df["canonical_smiles"] = llm_df["smiles"].swifter.apply(canonical_smiles)

patterns_df = results_from_patterns_df.copy()
patterns_df["label"] = patterns_df.swifter.apply(get_unique_label, axis=1)
patterns_df.set_index("label", inplace=True)
patterns_df["canonical_smiles"] = patterns_df["smiles"].swifter.apply(canonical_smiles)

Pandas Apply:   0%|          | 0/109129 [00:00<?, ?it/s]

Pandas Apply:   0%|          | 0/109129 [00:00<?, ?it/s]

Pandas Apply:   0%|          | 0/58989 [00:00<?, ?it/s]

Pandas Apply:   0%|          | 0/58989 [00:00<?, ?it/s]

In [15]:
data_block_files = list(DATA_BLOCKS_DIR.glob("**/*.json"))

num_entries = 0
num_combinations = {1: 0, 2: 0}

for data_block_file in tqdm(data_block_files, desc="Processing data block files"):
    if data_block_file.stat().st_size == 0:
        continue
    article_id = data_block_file.parent.name
    file_stem = data_block_file.stem
    with open(data_block_file, "r") as f:
        data_blocks = json.load(f)
    for data_block in data_blocks:
        if data_block["type"] != "rotation":
            continue

        combinations_file = (
            DATA_FOR_CURATION_DIR
            / article_id
            / file_stem
            / f"{data_block['start']}.json"
        )
        if not REPLACE_STORED_RESULTS and combinations_file.exists():
            continue

        label = f"{article_id}_{file_stem}_{data_block['start']}"

        molecules = {}
        original_smiles = {}
        excess = pd.NA
        excess_or_ratio = None

        for df in [llm_df, patterns_df]:
            if label in df.index:
                row = df.loc[label]
                canonical_smiles = row["canonical_smiles"]
                if canonical_smiles is not None:
                    molecules[canonical_smiles] = row["molecule"]
                    original_smiles[canonical_smiles] = row["smiles"]
                excess_value = float(row["excess"])
                if not np.isnan(excess_value):  # from patterns preferred if available
                    excess = excess_value
                    excess_or_ratio = row["excess_or_ratio"]

        if molecules and not pd.isna(excess):
            num_entries += 1
            combinations = [
                {
                    "molecule": molecules[canonical_smiles],
                    "smiles": original_smiles[canonical_smiles],
                    "excess": excess,
                    "excess_or_ratio": excess_or_ratio,
                }
                for canonical_smiles in molecules.keys()
            ]
            combinations_file.parent.mkdir(parents=True, exist_ok=True)
            with open(combinations_file, "w") as f:
                json.dump(combinations, f, indent=2)
            num_combinations[len(combinations)] += 1

print(f"Number of entries: {num_entries}")
for n, count in num_combinations.items():
    print(f"Number of {n}-combinations: {count}")
print(
    f"Total number of combinations: {sum(k * v for k, v in num_combinations.items())}"
)

Processing data block files:   0%|          | 0/12466 [00:00<?, ?it/s]

Number of entries: 96465
Number of 1-combinations: 91605
Number of 2-combinations: 4860
Total number of combinations: 101325
